In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawProduct"
silverTable = f"{catalog}.silver_schema_{env}.cleanedProduct"
print(f"Source Table : {bronzeTable}")
print(f"Target Table : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_{env}/vol_saleslake_src_files_{env}/daily_product/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    CAST(TRIM(product_id) AS INT)             AS product_id,
    UPPER(TRIM(sku))                          AS sku,
    UPPER(TRIM(product_name))                       AS product_name,
    UPPER(TRIM(category))                     AS category,
    UPPER(TRIM(sub_category))                 AS sub_category,
    UPPER(TRIM(brand))                        AS brand,
    UPPER(TRIM(supplier))                     AS supplier,
    CAST(TRIM(unit_cost) AS DECIMAL(18,2))    AS unit_cost,
    CAST(TRIM(list_price) AS DECIMAL(18,2))   AS list_price,
    TO_DATE(TRIM(launch_date), 'yyyy-MM-dd')  AS launch_date,
    UPPER(TRIM(status))                       AS status,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTable}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTable}
)
ORDER BY product_id
""")
print("Product Silver Load Completed Successfully")

In [0]:
# Product Row Count

spark.sql(f"""
SELECT 
    COUNT(*) AS product_count
FROM {silverTable}
""").show()

In [0]:
# Duplicate Product Check
# Expected result:
# No duplicate product_id

spark.sql(f"""
SELECT 
    product_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY product_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Null Product ID Check
# Expected result:

null_product_id = 0
spark.sql(f"""
SELECT 
    COUNT(*) AS null_product_id
FROM {silverTable}
WHERE product_id IS NULL
""").show()

In [0]:
# Invalid Price Check
# Unit cost and list price should not be negative.

spark.sql(f"""
SELECT 
    COUNT(*) AS invalid_price_records
FROM {silverTable}
WHERE unit_cost < 0
   OR list_price < 0
""").show()

In [0]:
# Sample Product Data

spark.sql(f"""
SELECT *
FROM {silverTable}
ORDER BY product_id
LIMIT 10
""").show(truncate=False)